# A2A Protocol Integration

This notebook demonstrates how agents communicate via the A2A protocol — discovery, task delegation, and response handling.

## 1. Load Environment

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)
print("✅ Environment loaded")

## 2. A2A Agent Discovery

The A2A protocol uses AgentCard for discovery. Each agent publishes its capabilities at `/.well-known/agent-card.json`.

In [ ]:
agent_urls = [
    "http://localhost:8101",  # doc_processor
    "http://localhost:8102",  # researcher
    "http://localhost:8103",  # writer
    "http://localhost:8104",  # reviewer
]

discovered_agents = []
for url in agent_urls:
    try:
        resp = httpx.get(f"{url}/.well-known/agent-card.json", timeout=5)
        if resp.status_code == 200:
            card = resp.json()
            discovered_agents.append({"url": url, "card": card})
            print(f"Discovered: {card.get('name', 'unknown')}")
            print(f"  URL: {url}")
            print(f"  Skills: {[s['name'] for s in card.get('skills', [])]}")
            print()
    except Exception as e:
        print(f"  ❌ {url}: {e}")

print(f"\nDiscovered {len(discovered_agents)} agents")
print("✅ Agent discovery complete")

## 3. A2A Message Format

A2A uses JSON-RPC 2.0 for communication. Here's the standard message format:

In [ ]:
# Standard A2A message/send request
a2a_request = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "unique-request-id",
    "params": {
        "message": {
            "role": "user",
            "parts": [
                {"kind": "text", "text": "Your message here"}
            ],
        }
    },
}

print("A2A Request Format:")
print(json.dumps(a2a_request, indent=2))

print("\n\nExpected A2A Response Format:")
a2a_response = {
    "jsonrpc": "2.0",
    "id": "unique-request-id",
    "result": {
        "artifacts": [{
            "parts": [{"kind": "text", "text": "Agent response here"}]
        }]
    },
}
print(json.dumps(a2a_response, indent=2))

## 4. Test Inter-Agent Communication

Simulate the orchestrator pattern: discover agents, then delegate a task to the researcher.

In [ ]:
def send_a2a_message(agent_url: str, message: str, request_id: str = "1") -> dict:
    """Send an A2A message and return the parsed response."""
    payload = {
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": request_id,
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": message}],
            }
        },
    }
    resp = httpx.post(agent_url, json=payload, timeout=120)
    return resp.json()


def extract_text(a2a_response: dict) -> str:
    """Extract text content from an A2A response."""
    if "result" in a2a_response:
        result = a2a_response["result"]
        for artifact in result.get("artifacts", []):
            for part in artifact.get("parts", []):
                if part.get("kind") == "text":
                    return part["text"]
        if "message" in result:
            for part in result["message"].get("parts", []):
                if part.get("kind") == "text":
                    return part["text"]
    return str(a2a_response)


# Test: Send research query to the researcher agent
print("Sending research query to Researcher agent via A2A...")
try:
    response = send_a2a_message(
        "http://localhost:8102",
        "What are the key findings in the indexed documents?"
    )
    text = extract_text(response)
    print(f"\nResearcher response (first 500 chars):")
    print(text[:500])
    print("\n✅ A2A inter-agent communication working")
except Exception as e:
    print(f"❌ Error: {e}")

## 5. Full Pipeline via Orchestrator

Send a research request to the Orchestrator, which coordinates all sub-agents.

In [ ]:
print("Sending research request to Orchestrator...")
print("(This triggers: plan → research → write → review)")
print()

try:
    response = send_a2a_message(
        "http://localhost:8100",
        "Provide a comprehensive analysis of the document processing techniques described in the indexed papers."
    )
    text = extract_text(response)
    print("Orchestrator final output:")
    print("=" * 60)
    print(text[:2000])
    if len(text) > 2000:
        print(f"\n... ({len(text) - 2000} more characters)")
    print("\n✅ Full multi-agent pipeline working via A2A")
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Ensure all agents are running: make agents-start")